In [2]:
import pandas as pd
import numpy as np

# Load the datasets
original_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Display basic information
print("Training data shape:", original_df.shape)
print("Test data shape:", test_df.shape)

print("\nTraining columns:")
print(original_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

print("\nFirst 5 rows of training data:")
display(original_df.head())

Training data shape: (9864, 19)
Test data shape: (2466, 18)

Training columns:
['Session_ID', 'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']

Test columns:
['Session_ID', 'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']

First 5 rows of training data:


,Session_ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,112163,0,0.00,0,0.0,3,44.500000,0.066667,0.133333,0.000000,0.0,Dec,2,2,9.0,3.0,Returning_Visitor,False,False
1,107490,0,0.00,0,0.0,12,460.200000,0.061111,0.111111,0.000000,0.0,Jul,2,2,6.0,4.0,Returning_Visitor,False,False
2,106273,4,48.80,0,0.0,11,344.800000,0.015385,0.054396,0.000000,0.0,Oct,3,2,1.0,4.0,Returning_Visitor,True,False
3,110651,0,0.00,0,0.0,23,517.035714,0.000000,0.009524,23.300007,0.0,Dec,4,2,8.0,2.0,New_Visitor,False,True
4,101259,7,110.25,0,0.0,20,266.583333,0.011111,0.039753,0.000000,0.0,Mar,2,2,1.0,2.0,Returning_Visitor,False,False


In [3]:
# Check missing values in the training dataset

missing_info = pd.DataFrame({
    "Missing_Count": original_df.isnull().sum(),
    "Missing_Percentage": (original_df.isnull().sum() / len(original_df)) * 100
})

missing_info = missing_info[missing_info["Missing_Count"] > 0]

print("Columns containing missing values:")
display(missing_info)

Columns containing missing values:


,Missing_Count,Missing_Percentage
Administrative_Duration,492,4.987835
ExitRates,691,7.005272
Region,591,5.991484
TrafficType,789,7.998783
VisitorType,394,3.994323


In [4]:
# Check data types of all columns
print("Column data types:")
display(original_df.dtypes)

# Check the target variable
print("\nRevenue distribution:")
display(original_df["Revenue"].value_counts())

print("\nRevenue percentage:")
display(original_df["Revenue"].value_counts(normalize=True) * 100)

Column data types:


Session_ID                   int64
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                     float64
TrafficType                float64
VisitorType                    str
Weekend                       bool
Revenue                       bool
dtype: object


Revenue distribution:


Revenue
False    8211
True     1653
Name: count, dtype: int64


Revenue percentage:


Revenue
False    83.242092
True     16.757908
Name: proportion, dtype: float64

In [6]:
# Separate features and target

X = original_df.drop(columns=["Revenue", "Session_ID"])

# Convert Revenue: False -> 0, True -> 1
y = original_df["Revenue"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (9864, 17)
y shape: (9864,)

Target distribution:
Revenue
0    8211
1    1653
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (7891, 17)
X_val: (1973, 17)
y_train: (7891,)
y_val: (1973,)


In [8]:
# Numerical columns
numerical_cols = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay"
]

# Categorical columns
categorical_cols = [
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Month",
    "Weekend"
]

print("Numerical columns:", len(numerical_cols))
print("Categorical columns:", len(categorical_cols))

Numerical columns: 10
Categorical columns: 7


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# Complete Logistic Regression pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [10]:
from sklearn.metrics import accuracy_score, classification_report

val_predictions = model.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, val_predictions))

print("\nClassification Report:")
print(classification_report(y_val, val_predictions))

Validation Accuracy: 0.8667004561581348

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.98      0.92      1642
           1       0.76      0.30      0.43       331

    accuracy                           0.87      1973
   macro avg       0.82      0.64      0.68      1973
weighted avg       0.86      0.87      0.84      1973



In [11]:
model.fit(X, y)

print("Model retrained on all 9864 training observations.")

Model retrained on all 9864 training observations.


In [12]:
X_test = test_df.drop(columns=["Session_ID"])

predictions = model.predict(X_test).astype(int)

print("Number of predictions:", len(predictions))
print("Prediction distribution:")
print(pd.Series(predictions).value_counts())

Number of predictions: 2466
Prediction distribution:
0    2276
1     190
Name: count, dtype: int64


In [13]:
X_processed = model.named_steps["preprocessor"].transform(X)

processed_df = pd.DataFrame(
    X_processed.toarray() if hasattr(X_processed, "toarray") else X_processed
)

print("processed_df shape:", processed_df.shape)
print("Missing values in processed_df:", processed_df.isnull().sum().sum())

processed_df shape: (9864, 75)
Missing values in processed_df: 0


In [14]:
import pandas as pd
import numpy as np

# ---------------------------------------------------
# Checkpoint information
# ---------------------------------------------------

original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (
    processed_rows / original_rows
) * 100


# ---------------------------------------------------
# Detect model used
# ---------------------------------------------------

final_model = model

# Handle GridSearchCV / RandomizedSearchCV
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_

# Handle sklearn Pipeline
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__


# ---------------------------------------------------
# Create checkpoint section
# ---------------------------------------------------

checkpoints = pd.DataFrame({

    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],

    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})


# ---------------------------------------------------
# Create prediction section
# ---------------------------------------------------

prediction_output = pd.DataFrame({

    "id": test_df["Session_ID"].astype(str),

    "value": np.asarray(predictions).astype(str)

})


# ---------------------------------------------------
# Combine and save
# ---------------------------------------------------

submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully.")
print(checkpoints)


submission.csv created successfully.
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  75
6  row_retained_percent               100.0
7            model_name  LogisticRegression
